# lab.py uniform-environment test

Exercises the `lab` API (`lab.py`) end to end -- declare, load, initiate a
run, run jobs by hand, verify -- to confirm it behaves the same whether this
notebook runs in JupyterLab (on the Modal container, volume mounted) or
locally in VS Code (no volume, everything routed through the deployed Modal
functions). See `lab.py`'s own module docstring for exactly what "the same"
means here: identical calls and return shapes, with one real difference --
locally, `lab.load()` comes back *unbound* (no bytes to read), so anything
that needs bound state (`.encode()`, running a job, binding a dataset) only
works from inside a container. Cells that need that are marked below and
check `lab._in_container()` themselves rather than failing confusingly.

Run this in JupyterLab first (everything should execute), then run it
unchanged from VS Code locally (the container-only cells will say so and
skip, everything else should behave identically).

## Setup

In [ ]:
import sys
from pathlib import Path

# notebooks/ is a folder down from the repo -- put the repo on the path so
# `import lab` works whether this is JupyterLab (already on path, this is a
# no-op) or a local VS Code kernel (not on path by default). Same pattern as
# load_tokenizer.ipynb.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import lab

IN_CONTAINER = lab._in_container()
print(f"in container (volume mounted): {IN_CONTAINER}")
print(f"ROOT: {lab.ROOT}")

## 1. Declare shared artifacts

A small, fast tree: one short public-domain text, one tiny tokenizer, one
tokenized source. Same shape as `tokenizer_demo.ipynb`, but through `lab.*`
instead of `dag.resolve` directly, and against the real volume instead of a
throwaway `.scratch/` root -- these are shared artifacts (no run_id), so
re-running this notebook from either side reuses whatever's already built
rather than redeclaring it.

In [ ]:
from sources.artifact import Source
from tokenizers.bpe import TokenizedSource, Tokenizer

romeojuliet = Source(
    name="romeojuliet", url="https://www.gutenberg.org/cache/epub/1513/pg1513.txt"
)

# vocab_size=256 -- distinct from declare.ipynb (350), init.ipynb (1420) and
# tokenizer_demo.ipynb (1000), so this test's own artifacts are easy to spot
# on the dashboard and don't collide with a real run's tokenizer.
tokenizer = Tokenizer(
    vocab_size=256,
    special_tokens=("<pad>", "<unk>"),
    sources=(romeojuliet,),
)
tokenized = TokenizedSource(tokenizer=tokenizer, source=romeojuliet)

print(f"tokenizer path: {tokenizer.artifact_path}")
print(f"tokenized path: {tokenized.artifact_path}")

In [ ]:
# Preview: reconciles the whole tree against the volume, writes nothing.
# In a container this returns the Declaration itself (.rows/.problems/.ok);
# locally it routes through the deployed `declare` function and returns None
# -- the same report, already printed either way.
report = lab.check(tokenized)
if report is not None:
    print(report)

In [ ]:
# Now actually declare: writes manifests for anything still `new`, commits.
# Same None-in-container-vs-Declaration-locally split as check().
result = lab.declare(tokenized)
if result is not None:
    print(result)

## 2. Load an artifact back, inspect its manifest

`lab.load(path)` by path -- same call in both environments. What comes back
differs: bound (usable -- `.vocab`, `.encode(...)`) in a container, unbound
(parameters and dependencies only) locally, per `lab.load`'s own docstring.

In [ ]:
loaded = lab.load(str(tokenizer.artifact_path))

print(loaded)                 # every parameter, read back off the manifest
print(loaded.commit)          # the code it was built under
print(loaded.deps())          # the artifacts underneath it (the source, here)
print(f"bound: {loaded.bound}")

## 3. Initiate a run

Same shape as `declare.ipynb`/`init.ipynb`: a fresh `run_id`, a dataset, a
pretraining config -- except the mock model family (no GPU needed, no real
training, just a fast simulated loss curve) and no `allocated_resources`,
since this is a cheap correctness test, not a real run. `RUN_ID` is
timestamp-free on purpose so re-running this cell doesn't pile up runs --
change it if you want a fresh one.

In [ ]:
from mappeddatasets.artifact import MappedDataSet
from models.mock.artifact import ModelParameters, Pretraining, PretrainingConfig

RUN_ID = "RUN_LAB_TEST_UNIFORM"  # fixed -- reruns reuse this run, don't pile up new ones

mapped = MappedDataSet.from_sources(
    tokenizer=tokenizer, train_sources=[romeojuliet], valid_sources=[romeojuliet]
)

pretraining = Pretraining(
    run_id=RUN_ID,
    dataset=mapped,
    tokenizer=tokenizer,
    model_parameters=ModelParameters(hidden_size=16, num_layers=1),
    config=PretrainingConfig(
        total_steps=50, batch_size=8, lr=1e-3, seed=1, checkpoint_every=25
    ),
)

report = lab.check(pretraining)
if report is not None:
    print(report)

In [ ]:
result = lab.declare(pretraining)
if result is not None:
    print(result)

## 4. Run jobs manually (container-only)

Runs every job the plan needs, in dependency order, using `lab.worker` --
the fake worker `lab.py` builds for exactly this: running a job by hand from
a cell, outside the run context (a real `run_job` call) it would otherwise
run under. No lease, no heartbeat, nothing to supersede it -- see
`lab.worker`'s own docstring in `lab.py`.

This needs bytes on disk to read and write, so it only runs in a container.
Locally, this cell reports that and does nothing -- from VS Code, the way to
actually build these artifacts is to declare them (done above, on the real
volume already) and launch the run from JupyterLab or the dashboard.

In [ ]:
import dag.resolve as dag_resolve

if IN_CONTAINER:
    for target in (tokenized, pretraining):
        for job in dag_resolve.job_list(dag_resolve.resolve(target)):
            if dag_resolve.status(job.artifact, lab.ROOT) == "done":
                print(f"already done: {job.artifact.artifact_path}")
                continue
            print(f"running {type(job).__name__} for {job.artifact.artifact_path}")
            job.run(lab.ROOT, lab.worker)
    lab.publish()  # land what these jobs just wrote before anything re-checks status
else:
    print("skipping -- no volume mounted here; declare (done above) reaches the "
          "volume fine, but building needs a container. Launch this run from "
          "JupyterLab or the dashboard to actually build it.")

## 5. Verify, bind, use

`lab.refresh()` first, in case something else built these since this session
started -- no-op locally (nothing to refresh). Then re-check status, and if
we're in a container, bind and actually use both artifacts: encode text with
the tokenizer, read back the run's progress file.

In [ ]:
lab.refresh()

report = lab.check(tokenized)
if report is not None:
    print(report)

report = lab.check(pretraining)
if report is not None:
    print(report)

In [ ]:
if IN_CONTAINER:
    bound_tokenizer = tokenizer.bind(lab.ROOT)
    print(f"{len(bound_tokenizer.vocab)} vocab entries")

    text = "But soft, what light through yonder window breaks?"
    ids = bound_tokenizer.encode(text)
    print(ids)
    print(repr(bound_tokenizer.decode(ids)))

    progress_path = pretraining.paths(lab.ROOT)["progress"]
    if progress_path.exists():
        print(progress_path.read_text())
    else:
        print("no progress file yet -- run part 4 first")
else:
    print("skipping bind/encode -- no local bytes to bind to here; this is the "
          "one place environment genuinely matters (see lab.py's module docstring)")

## 6. Publish

Land anything this session wrote (part 4 already published once, but this is
idempotent and harmless to call again -- a no-op locally, same as `refresh`).

In [ ]:
lab.publish()